In [6]:
import cplex
import pandas as pd

# 讀取資料
df_cart = pd.read_excel("data.xlsx", sheet_name="CART")
df_wip  = pd.read_excel("data.xlsx", sheet_name="WIP")
df_xfer = pd.read_excel("data.xlsx", sheet_name="XFER_TIME")

# 建立 XFER_TIME 快速查表
xfer_dict = {}
for _, row in df_xfer.iterrows():
    src = row["FROM"]
    dst = row["TO"]
    xfer_dict.setdefault(src, {})[dst] = row["XFER_TIME"]

# 參數設定
PENALTY  = 10000   # 未接受訂單 penalty
CAPACITY = 2       # 每台 CART 最大載貨量
M        = 1e5     # 大M值

# 建立集合
carts  = df_cart["CART_ID"].tolist()
orders = df_wip["WIP_ID"].tolist()

# 節點定義
# 每筆訂單 i 對應取貨節點 "P_i" 與送貨節點 "D_i"
nodes = []
for o in orders:
    nodes.append("P_" + o)
    nodes.append("D_" + o)
# 每台 CART 建立起點 S_c 及終點 E_c
for c in carts:
    nodes.append("S_" + c)
    nodes.append("E_" + c)

# 節點對應位置
node_location = {}
for c in carts:
    loc = df_cart.loc[df_cart["CART_ID"]==c, "INIT_LOC"].values[0]
    node_location["S_" + c] = loc
    node_location["E_" + c] = loc
for o in orders:
    pickup_loc   = df_wip.loc[df_wip["WIP_ID"]==o, "FROM"].values[0]
    delivery_loc = df_wip.loc[df_wip["WIP_ID"]==o, "TO"].values[0]
    node_location["P_" + o] = pickup_loc
    node_location["D_" + o] = delivery_loc

# 建立 travel 時間矩陣 (若找不到資料則用 M/10)
travel = {}
for i in nodes:
    travel[i] = {}
    for j in nodes:
        if i == j:
            travel[i][j] = 0
        else:
            loc_i = node_location[i]
            loc_j = node_location[j]
            travel[i][j] = xfer_dict.get(loc_i, {}).get(loc_j, M/10)

# Q_time：各訂單送貨節點要求的完成時間
Q_time = {}
for o in orders:
    Q_time["D_" + o] = df_wip.loc[df_wip["WIP_ID"]==o, "Remaining Q-Time"].values[0]

# -------------------------------
# 建立 CPLEX 模型
cpx = cplex.Cplex()
cpx.objective.set_sense(cpx.objective.sense.minimize)

# -------------------------------
# 1. 決策變數
# (a) x[c,i,j]，表示 CART c 從節點 i 到節點 j 是否走此弧（二元變數）
x_names = []
x_obj   = []
for c in carts:
    for i in nodes:
        for j in nodes:
            if i != j:
                varname = f"x_{c}_{i}_{j}"
                x_names.append(varname)
                x_obj.append(travel[i][j])
cpx.variables.add(names=x_names, obj=x_obj,
                    types=[cpx.variables.type.binary]*len(x_names))

# (b) T[i]：到達節點 i 的時間（連續變數，lb=0）
T_names = ["T_" + i for i in nodes]
T_obj   = [0.0]*len(nodes)
T_lb    = [0.0]*len(nodes)
cpx.variables.add(names=T_names, obj=T_obj, lb=T_lb)

# (c) L[i]：到達節點 i 後的載貨量（連續變數，lb=0, ub=CAPACITY）
L_names = ["L_" + i for i in nodes]
L_obj   = [0.0]*len(nodes)
L_lb    = [0.0]*len(nodes)
L_ub    = [CAPACITY]*len(nodes)
cpx.variables.add(names=L_names, obj=L_obj, lb=L_lb, ub=L_ub)

# (d) a[o]：訂單 o 是否被接受（二元變數）
a_names = ["a_" + o for o in orders]
a_obj   = [ -PENALTY ]*len(orders)  
# 目標函數會計 PENALTY*(1 - a[o]) 效果，因此 a 變數設為 -PENALTY 係數
cpx.variables.add(names=a_names, obj=a_obj,
                    types=[cpx.variables.type.binary]*len(orders))

# 補充：
# 原目標設定為 minimize sum_{c,i,j} travel[i][j]*x[c,i,j] + sum_{o} (1-a[o])*PENALTY
# 這裡以變數 a 係數為 -PENALTY，等價於 minimize sum(travel*x) - PENALTY* a, 加上一個常數 PENALTY*|orders|

# -------------------------------
# 2. 建立約束式
constr_names = []
constr_expr  = []
constr_sense = []
constr_rhs   = []

# (1) 每台 CART 必須從自己的起點出發且到達自己的終點
for c in carts:
    # 從起點 S_c 出發的弧數 = 1
    expr_vars = []
    expr_coefs = []
    start_node = "S_" + c
    for j in nodes:
        if j != start_node:
            varname = f"x_{c}_{start_node}_{j}"
            if varname in x_names:
                expr_vars.append(varname)
                expr_coefs.append(1.0)
    constr_names.append(f"start_{c}")
    constr_expr.append([expr_vars, expr_coefs])
    constr_sense.append("E")
    constr_rhs.append(1.0)
    
    # 進入終點 E_c 的弧數 = 1
    expr_vars = []
    expr_coefs = []
    end_node = "E_" + c
    for i in nodes:
        if i != end_node:
            varname = f"x_{c}_{i}_{end_node}"
            if varname in x_names:
                expr_vars.append(varname)
                expr_coefs.append(1.0)
    constr_names.append(f"end_{c}")
    constr_expr.append([expr_vars, expr_coefs])
    constr_sense.append("E")
    constr_rhs.append(1.0)

# (2) 若訂單 o 被接受 (a[o]=1) 則其取貨與送貨節點各被走訪一次
for o in orders:
    # 取貨節點 P_o 必須有流入= a[o]
    expr_vars = []
    expr_coefs = []
    pickup = "P_" + o
    for c in carts:
        for i in nodes:
            if i != pickup:
                varname = f"x_{c}_{i}_{pickup}"
                if varname in x_names:
                    expr_vars.append(varname)
                    expr_coefs.append(1.0)
    constr_names.append(f"pickup_{o}")
    constr_expr.append([expr_vars, expr_coefs])
    constr_sense.append("E")
    # 右側設定為 a[o] （非線性寫法，但這裡可近似設 1，再與 a[o] 其他約束聯立調整）
    # 這裡以 1 表示：若不接受，流量自然為0；詳細表示可用大M分段處理
    constr_rhs.append(1.0)
    
    # 送貨節點 D_o
    expr_vars = []
    expr_coefs = []
    delivery = "D_" + o
    for c in carts:
        for i in nodes:
            if i != delivery:
                varname = f"x_{c}_{i}_{delivery}"
                if varname in x_names:
                    expr_vars.append(varname)
                    expr_coefs.append(1.0)
    constr_names.append(f"delivery_{o}")
    constr_expr.append([expr_vars, expr_coefs])
    constr_sense.append("E")
    constr_rhs.append(1.0)

# (3) 流平衡約束：對於每個 CART 及非起終點節點，流入 = 流出
for c in carts:
    for n in nodes:
        if n.startswith("S_") and n.endswith(c): continue
        if n.startswith("E_") and n.endswith(c): continue
        expr_vars_in = []
        expr_coefs_in = []
        expr_vars_out = []
        expr_coefs_out = []
        for i in nodes:
            if i != n:
                varname = f"x_{c}_{i}_{n}"
                if varname in x_names:
                    expr_vars_in.append(varname)
                    expr_coefs_in.append(1.0)
        for j in nodes:
            if j != n:
                varname = f"x_{c}_{n}_{j}"
                if varname in x_names:
                    expr_vars_out.append(varname)
                    expr_coefs_out.append(1.0)
        # 將流入-流出 = 0
        constr_names.append(f"flow_{c}_{n}")
        all_vars = expr_vars_in + expr_vars_out
        all_coefs = expr_coefs_in + [-coef for coef in expr_coefs_out]
        constr_expr.append([all_vars, all_coefs])
        constr_sense.append("E")
        constr_rhs.append(0.0)

# (4) 時間連續性約束
for c in carts:
    for i in nodes:
        for j in nodes:
            if i != j:
                varname = f"x_{c}_{i}_{j}"
                if varname in x_names:
                    # T[j] - T[i] >= travel[i][j] - M*(1 - x[c,i,j])
                    # 改寫成： T[j] - T[i] - M*x[c,i,j] >= travel[i][j] - M
                    constr_names.append(f"time_{c}_{i}_{j}")
                    constr_expr.append([["T_"+j, "T_"+i, varname],
                                         [1.0, -1.0, M]])
                    constr_sense.append("G")
                    constr_rhs.append(travel[i][j] - M)

# (5) 取貨必在送貨之前：對每筆訂單
for o in orders:
    pickup = "P_" + o
    delivery = "D_" + o
    constr_names.append(f"order_{o}")
    constr_expr.append([["T_"+delivery, "T_"+pickup],
                         [1.0, -1.0]])
    constr_sense.append("G")
    constr_rhs.append(travel[pickup][delivery])  # 至少需要運行時間

# (6) 載貨量更新（僅示範基本概念，取貨點 +1，送貨點 -1）
for c in carts:
    for i in nodes:
        for j in nodes:
            if i != j:
                varname = f"x_{c}_{i}_{j}"
                if varname in x_names:
                    # 若 i 為取貨點則貨量 +1；若為送貨點則 -1；否則變化0
                    coef = 0
                    if i.startswith("P_"): coef = 1
                    elif i.startswith("D_"): coef = -1
                    constr_names.append(f"load_{c}_{i}_{j}")
                    constr_expr.append([["L_"+j, "L_"+i, varname],
                                        [1.0, -1.0, M*coef]])
                    constr_sense.append("G")
                    constr_rhs.append(coef - M)
# (7) 時間視窗：若訂單 o 被接受，送貨節點 T <= Q_time
for o in orders:
    delivery = "D_" + o
    constr_names.append(f"timeWindow_{o}")
    constr_expr.append([["T_"+delivery, "a_"+o],
                        [1.0, M]])  # 使用大M處理非接受情形
    constr_sense.append("L")
    constr_rhs.append(Q_time[delivery] + M)
    
# -------------------------------
# 將所有約束加入模型
cpx.linear_constraints.add(names=constr_names,
                           lin_expr=constr_expr,
                           senses=constr_sense,
                           rhs=constr_rhs)

# -------------------------------
# 求解模型
cpx.solve()
print("目標值 = ", cpx.solution.get_objective_value())

# 列出部分 x 變數解
sol = cpx.solution
for var in x_names:
    if sol.get_values(var) > 0.5:
        print(var, "=", sol.get_values(var))

# -------------------------------
# 額外輸出指標

# 1. 計算淘汰的訂單數 (若 a[o] < 0.5 則視為淘汰)
rejected_count = 0
for o in orders:
    a_val = sol.get_value("a_" + o)
    if a_val < 0.5:  # 若未接受
        rejected_count += 1
print("\n淘汰的訂單數:", rejected_count)

# 2. 計算總行駛距離：累計所有 x 變數中取1的 travel cost
total_distance = 0.0
x_values = sol.get_values(x_names)
for i, x_val in enumerate(x_values):
    if x_val > 0.5:
        total_distance += x_obj[i]
print("總行駛距離:", total_distance)

# 3. 計算全部訂單完成耗時：各 CART 終點（E_c）的 T 值取最大
max_completion_time = 0.0
for c in carts:
    t_val = sol.get_value("T_E_" + c)
    if t_val > max_completion_time:
        max_completion_time = t_val
print("全部訂單完成耗時(最後一台 CART 完成時間):", max_completion_time)

KeyboardInterrupt: 